# Amazon Top-20 — SGL · NCL 2차 확장 Ablation + 최종 제출

**팀 공통 Validation 분할은 수정하지 않습니다.** Kaggle Input에 (1) 대회 원본 5개 파일, (2) 기존 Private 팀 공통 키트(`amazon_topk_teamkit_20261110.zip`에서 만든 데이터셋)를 연결하세요. GPU T4 한 장을 사용합니다.

지난 QUICK 실험에서 확인한 결과 (공통 split `f0dfad2f4431a876`):

| 실험 | Recall@20 | 최고 Epoch |
|---|---:|---:|
| S00 (SSL 없음 = LightGCN 기준선) | 0.034391 | 700 |
| S01 (Edge 0.1, SSL 0.02) | 0.033282 | 500 |
| N01 (Structural만) | 0.021760 | 700 |
| N02 (Structural + Proto 0.002) | 0.013003 | 700 |
| 이전 별도 LightGCN 최고 (참고값) | 0.036055 | 575 |

### 이번 설계
- **SGL:** 0.02보다 작은 SSL 가중치, edge dropout, InfoNCE temperature, 128차원·학습률 조합, 학습 길이
- **NCL:** 700 Epoch 이후 추세, structural weight, 128차원, 얕은 그래프, 더 약한 prototype loss, clustering과 warmup
- **비교 기준:** 공통 `val_train.csv` → 모든 모델이 같은 `val_truth.csv`로 **pooled Recall@20** 계산
- **최종 제출:** 이번 실험 결과 중 선택한 모델 설정으로 원본 `train.csv` **전체 재학습** → `test.csv` 예측 → `submission.csv` 검사

> **주의:** 여기의 SGL-ED와 NCL은 논문 아이디어를 구현한 학습용 PyTorch 버전입니다. NCL의 프로토타입은 GPU spherical k-means를 사용하므로 공식 RecBole/FAISS 재현 구현과 동일하지 않습니다. `SSL=0`인 SGL은 **SGL이 아니라 LightGCN 기준선**입니다. Validation 정답은 모델 선택에만 사용하며 최종 학습에는 포함된 원본 `train.csv` 전체를 사용합니다.

**실행:** 위에서 아래로 Run All. 많은 실험으로 GPU 시간 제한에 걸리면 `RUN_PROFILE`을 `QUICK` 또는 `BALANCED`로 바꾸어 재시작하세요. 각 실험 결과는 완료 직후 파일로 저장됩니다.

## 1. GPU와 대회 원본 데이터 경로 확인

In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys, zipfile, time
import torch

WORK = Path('/kaggle/working')
WORK.mkdir(parents=True, exist_ok=True)
OUT = WORK / 'sgl_ncl_extended'
OUT.mkdir(parents=True, exist_ok=True)
INPUT = Path('/kaggle/input')
roots = sorted({p.parent for p in INPUT.rglob('train.csv')
                if (p.parent/'test.csv').exists()})
if len(roots) != 1:
    raise RuntimeError(f'대회 원본 train.csv/test.csv 연결을 확인하세요. 찾은 경로: {roots}')
DATA = roots[0]
TRAIN = DATA/'train.csv'
TEST = DATA/'test.csv'
VALID = WORK/'validation'
SPLIT_ID = 'f0dfad2f4431a876'
SEED = 42
DEVICE = 'cuda:0'
if not torch.cuda.is_available():
    raise RuntimeError('GPU가 꺼져 있어요. Kaggle Settings → Accelerator → GPU 선택 후 실행하세요.')
print('Original dataset:', DATA)
print('GPU:', torch.cuda.get_device_name(0), '|', DEVICE)
print('Output directory:', OUT)

## 2. 팀 공통 평가 함수 (`teamkit.py`) 저장 — 기존 키트와 동일

In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)

## 3. 그래프 임베딩·BPR·예측 공통 코드 (`graph_models.py`) 저장

In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()

## 4. SGL·NCL 모델 학습 코드 (`sgl_ncl_runner.py`) 저장 — 기존 QUICK 코드와 동일

In [ ]:
%%writefile /kaggle/working/sgl_ncl_runner.py
"""LightGCN backbone with SGL-ED or NCL-style SSL, for the existing team kit.

Faithfulness note:
- SGL-ED: independently edge-subsampled, re-normalized graph views, InfoNCE.
- NCL: layer-0 versus 2h-hop structural InfoNCE, user/item prototypes from
  detached spherical torch k-means; not bit-for-bit identical to RecBole/FAISS.
Validation truth is used strictly for model selection and reporting.
"""
from __future__ import annotations
import argparse
import copy
import json
import math
import random
import time
from pathlib import Path

import torch
import torch.nn.functional as F

from graph_models import (
    LightGCN, _training_loss, load_data, load_truth, normalized_graph,
    predict_top20, recall_at_20, sample_triples, write_predictions,
)


def edge_view(positive_users, positive_items, n_users, n_items, drop, device):
    """Independent sampled bipartite graph, symmetric, degree re-normalized."""
    if not 0 <= drop < 1:
        raise ValueError('drop ratio must be in [0,1)')
    keep = torch.rand(positive_users.numel(), device=device) >= drop
    u, i = positive_users[keep], positive_items[keep] + n_users
    row = torch.cat((u, i))
    col = torch.cat((i, u))
    counts = torch.bincount(row, minlength=n_users + n_items).float()
    values = counts[row].clamp_min(1).rsqrt() * counts[col].clamp_min(1).rsqrt()
    return torch.sparse_coo_tensor(
        torch.stack((row, col)), values,
        (n_users + n_items, n_users + n_items), device=device
    ).coalesce()


def propagation(model, graph, depth):
    curr = torch.cat((model.user_embedding.weight, model.item_embedding.weight), dim=0)
    steps = [curr]
    for _ in range(depth):
        curr = torch.sparse.mm(graph, curr)
        steps.append(curr)
    mean = torch.stack(steps[:model.layers + 1], 0).mean(0)
    n_users = model.user_embedding.num_embeddings
    return mean[:n_users], mean[n_users:], steps


def nce_full(anchor, positive, candidate_all, tau):
    """Sampled anchor nodes, ALL same-type nodes used in InfoNCE denominator."""
    anchor = F.normalize(anchor, dim=-1)
    positive = F.normalize(positive, dim=-1)
    candidate_all = F.normalize(candidate_all, dim=-1)
    logits = anchor @ candidate_all.T / tau
    pos = (anchor * positive).sum(-1) / tau
    return (torch.logsumexp(logits, dim=1) - pos).mean()


def sgl_info_nce(view1, view2, sampled_users, sampled_items, n_users, tau):
    loss_u = nce_full(
        view1[0][sampled_users], view2[0][sampled_users], view2[0], tau
    )
    loss_i = nce_full(
        view1[1][sampled_items], view2[1][sampled_items], view2[1], tau
    )
    return loss_u + loss_i


@torch.no_grad()
def spherical_kmeans(x, n_clusters, iterations=5):
    """GPU spherical k-means to avoid requiring faiss-gpu/RecBole."""
    x = F.normalize(x.detach(), dim=1)
    n = x.size(0)
    if n_clusters > n:
        raise ValueError('Clusters exceed node count')
    centers = x[torch.randperm(n, device=x.device)[:n_clusters]].clone()
    for _ in range(iterations):
        labels = (x @ centers.T).argmax(dim=1)
        sums = torch.zeros_like(centers)
        sums.index_add_(0, labels, x)
        counts = torch.bincount(labels, minlength=n_clusters)
        empty = torch.where(counts == 0)[0]
        if empty.numel():
            sums[empty] = x[torch.randint(n, (empty.numel(),), device=x.device)]
        centers = F.normalize(sums, dim=1)
    labels = (x @ centers.T).argmax(dim=1)
    return centers.detach(), labels.detach()


def prototype_nce(raw_users, raw_items, sample_users, sample_items,
                  user_clusters, item_clusters, tau):
    cent_u, lab_u = user_clusters
    cent_i, lab_i = item_clusters
    u = F.normalize(raw_users[sample_users], dim=-1)
    i = F.normalize(raw_items[sample_items], dim=-1)
    lu = F.cross_entropy(u @ cent_u.T / tau, lab_u[sample_users])
    li = F.cross_entropy(i @ cent_i.T / tau, lab_i[sample_items])
    return lu + li


def evaluate(model, graph, data, truth, eval_batch):
    predictions = predict_top20(model, graph, data, eval_batch)
    return recall_at_20(predictions, truth)


def train_run(a):
    if a.epochs < 1 or a.layers < 1 or a.dim < 1 or a.samples_per_epoch < 1:
        raise ValueError('epochs/layers/dim/samples_per_epoch must be positive')
    if a.ssl_nodes < 1 or a.eval_every < 1 or a.tau <= 0 or a.lr <= 0:
        raise ValueError('ssl_nodes/eval_every/tau/lr must be positive')
    if a.model == 'ncl' and (a.hyper_layers < 1 or a.clusters < 2 or a.cluster_every < 1):
        raise ValueError('invalid NCL clustering/hyper-layer configuration')
    if a.ssl_weight < 0 or a.proto_weight < 0 or a.reg < 0:
        raise ValueError('loss weights must be nonnegative')
    if a.device.startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but not available')
    device = torch.device(a.device)
    random.seed(a.seed)
    torch.manual_seed(a.seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(a.seed)
    rng = random.Random(a.seed)
    data = load_data(a.fit, a.users)
    truth = load_truth(a.truth, data.target_users) if a.truth else None
    graph = normalized_graph(data, device)
    edge_u = torch.tensor(data.positive_users, dtype=torch.long, device=device)
    edge_i = torch.tensor(data.positive_items, dtype=torch.long, device=device)
    model = LightGCN(data.num_users, data.num_items, a.dim, a.layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=a.lr)
    print(json.dumps({'info':'START', 'model':a.model, 'users':data.num_users, 'items':data.num_items,
                      'fit_edges':data.num_edges,'layers':a.layers,'dim':a.dim,
                      'device':str(device),'epochs':a.epochs, 'ssl_weight':a.ssl_weight,
                      'proto_weight':a.proto_weight}), flush=True)
    best_score = -1.0
    best_epoch = None
    best_state = None
    stale_evals = 0
    current_clusters = None
    last_loss = None
    trained_epochs = 0
    out_path = Path(a.out)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, a.epochs + 1):
        tick = time.monotonic()
        trained_epochs = epoch
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, a.samples_per_epoch, rng, device)
        anchor_u = torch.unique(sampled[0][:a.ssl_nodes])
        anchor_i = torch.unique(sampled[1][:a.ssl_nodes])
        if a.model == 'ncl':
            u_vec, i_vec, steps = propagation(model, graph, max(a.layers, 2 * a.hyper_layers))
        else:
            u_vec, i_vec = model(graph)
            steps = None
        bpr_reg, bpr, reg = _training_loss(model, u_vec, i_vec, sampled, a.reg)
        ssl = bpr_reg.new_zeros(())
        proto = bpr_reg.new_zeros(())
        if a.model == 'sgl' and a.ssl_weight > 0:
            g1 = edge_view(edge_u, edge_i, data.num_users, data.num_items, a.drop, device)
            g2 = edge_view(edge_u, edge_i, data.num_users, data.num_items, a.drop, device)
            e1 = model(g1)
            e2 = model(g2)
            ssl = sgl_info_nce(e1, e2, anchor_u, anchor_i, data.num_users, a.tau)
        elif a.model == 'ncl':
            raw = steps[0]
            context = steps[2 * a.hyper_layers]
            raw_user = raw[:data.num_users]
            raw_item = raw[data.num_users:]
            ctx_user = context[:data.num_users]
            ctx_item = context[data.num_users:]
            if a.ssl_weight > 0:
                ssl = (nce_full(ctx_user[anchor_u], raw_user[anchor_u], raw_user, a.tau)
                       + a.alpha * nce_full(ctx_item[anchor_i], raw_item[anchor_i], raw_item, a.tau))
            if a.proto_weight > 0 and epoch >= a.warmup:
                if current_clusters is None or (epoch - a.warmup) % a.cluster_every == 0:
                    current_clusters = (
                        spherical_kmeans(raw_user, a.clusters),
                        spherical_kmeans(raw_item, a.clusters),
                    )
                proto = prototype_nce(raw_user, raw_item, anchor_u, anchor_i,
                                      current_clusters[0], current_clusters[1], a.tau)
        loss = bpr_reg + a.ssl_weight * ssl + a.proto_weight * proto
        if not torch.isfinite(loss):
            raise RuntimeError(f'Loss non-finite at epoch {epoch}: {float(loss)}')
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach())
        record = {'epoch':epoch, 'loss':round(last_loss,6),
                  'bpr':round(float(bpr.detach()),6),
                  'reg':round(float(reg.detach()),6),
                  'ssl':round(float(ssl.detach()),6),
                  'proto':round(float(proto.detach()),6),
                  'seconds':round(time.monotonic()-tick,3)}
        del sampled, u_vec, i_vec, steps, loss, bpr_reg, bpr, reg, ssl, proto
        should_eval = truth is not None and (epoch == 1 or epoch % a.eval_every == 0 or epoch == a.epochs)
        if should_eval:
            score = evaluate(model, graph, data, truth, a.eval_batch)
            record.update(score)
            if score['recall_at_20'] > best_score:
                best_score = score['recall_at_20']
                best_epoch = epoch
                best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                stale_evals = 0
                record['best'] = True
            else:
                stale_evals += 1
                record['best'] = False
        elif epoch % 50 != 0:
            continue  # emit concise logs only, not 700+ lines per run
        print(json.dumps(record), flush=True)
        if should_eval and a.patience > 0 and stale_evals >= a.patience:
            print(json.dumps({'early_stop_epoch':epoch}),flush=True)
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    predictions = predict_top20(model, graph, data, a.eval_batch)
    write_predictions(out_path, data.target_users, predictions)
    result = {
        'model':a.model, 'output':str(out_path), 'best_epoch':best_epoch,
        'trained_epochs':trained_epochs,'epochs_requested':a.epochs,'device':str(device),
        'layers':a.layers,'dim':a.dim,'lr':a.lr,'reg':a.reg,'ssl_weight':a.ssl_weight,
        'proto_weight':a.proto_weight,'drop':a.drop,'tau':a.tau,'clusters':a.clusters,
        'warmup':a.warmup,'hyper_layers':a.hyper_layers,'alpha':a.alpha,
        'samples_per_epoch':a.samples_per_epoch,'seed':a.seed,'last_loss':last_loss,
    }
    if truth is not None:
        result.update(recall_at_20(predictions,truth))
    if a.metrics:
        Path(a.metrics).parent.mkdir(parents=True,exist_ok=True)
        Path(a.metrics).write_text(json.dumps(result,indent=2),encoding='utf-8')
    if a.checkpoint:
        torch.save({'model':a.model,'weights':model.state_dict(),'config':vars(a)},a.checkpoint)
    print(json.dumps({'final':result}),flush=True)
    return result


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--model',choices=['sgl','ncl'],required=True)
    p.add_argument('--fit',required=True)
    p.add_argument('--users',required=True)
    p.add_argument('--truth',default=None)
    p.add_argument('--out',required=True)
    p.add_argument('--metrics',default=None)
    p.add_argument('--checkpoint',default=None)
    p.add_argument('--device',default='cuda:0')
    p.add_argument('--epochs',type=int,default=700)
    p.add_argument('--eval-every',type=int,default=25)
    p.add_argument('--patience',type=int,default=0)
    p.add_argument('--eval-batch',type=int,default=256)
    p.add_argument('--dim',type=int,default=64)
    p.add_argument('--layers',type=int,default=3)
    p.add_argument('--lr',type=float,default=.002)
    p.add_argument('--reg',type=float,default=.0001)
    p.add_argument('--samples-per-epoch',type=int,default=100000)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--ssl-nodes',type=int,default=192)
    p.add_argument('--ssl-weight',type=float,default=.05)
    p.add_argument('--drop',type=float,default=.1)
    p.add_argument('--tau',type=float,default=.2)
    p.add_argument('--proto-weight',type=float,default=.005)
    p.add_argument('--clusters',type=int,default=64)
    p.add_argument('--cluster-every',type=int,default=50)
    p.add_argument('--warmup',type=int,default=50)
    p.add_argument('--hyper-layers',type=int,default=1)
    p.add_argument('--alpha',type=float,default=1.)
    a=p.parse_args()
    train_run(a)

if __name__=='__main__':
    main()

## 5. 팀 공통 Validation 데이터 로드·해시 검사 (분할 재생성하지 않음)

In [ ]:
sys.path.insert(0, str(WORK))
from teamkit import verify_split, read_truth

required = ('val_train.csv','val_truth.csv','val_users.csv','manifest.json')
source_folder = None
source_zip = None
for base in (INPUT, WORK):
    for mf in base.rglob('manifest.json'):
        folder = mf.parent
        if folder == VALID or not all((folder/n).is_file() for n in required):
            continue
        try:
            obj = json.loads(mf.read_text(encoding='utf-8'))
        except (OSError,ValueError):
            continue
        if obj.get('split_id') == SPLIT_ID:
            source_folder = folder
            break
    if source_folder:
        break
if not source_folder:
    for base in (INPUT, WORK):
        hits = list(base.rglob('amazon_topk_teamkit_20261110.zip'))
        if hits:
            source_zip = hits[0]
            break
if source_folder is None and source_zip is None:
    raise FileNotFoundError('기존 팀 공통 ZIP을 오른쪽 Input에서 Add Input으로 연결해 주세요.')
VALID.mkdir(parents=True, exist_ok=True)
if source_folder is not None:
    for n in required:
        shutil.copyfile(source_folder/n, VALID/n)
    print('Team ZIP read from:', source_folder)
else:
    with zipfile.ZipFile(source_zip) as z:
        for n in required:
            paths = [x for x in z.namelist() if x.endswith('/validation/'+n)]
            if len(paths) != 1:
                raise ValueError(f'공통 ZIP 파일 {n}을 정확히 한 개 찾지 못했습니다: {paths}')
            with z.open(paths[0]) as src, (VALID/n).open('wb') as dst:
                shutil.copyfileobj(src, dst)
    print('Team ZIP extracted from:', source_zip)
manifest = verify_split(VALID)
assert manifest['split_id'] == SPLIT_ID, '팀 공통 split ID가 다릅니다.'
h = hashlib.sha256()
with TRAIN.open('rb') as f:
    for chunk in iter(lambda:f.read(1024*1024), b''):
        h.update(chunk)
assert h.hexdigest() == manifest['source_train_sha256'], '원본 train.csv가 팀 공통 분할과 일치하지 않습니다.'
print('Validation matched:', manifest['validation_users'],'users,',manifest['fit_rows'],'train edges,',manifest['truth_rows'],'truth edges')
print('Split ID:',manifest['split_id'])

## 6. 실험 규모 및 후보 설정

- `QUICK`: 8개 핵심 후보, 다음 실험 방향을 보는 용도
- `BALANCED`: 21개 내외 주요 후보 (권장)
- `MAX`: 아래 전체 42개 후보 (시간과 GPU 사용량이 많음)

**모든 후보를 노트북에 포함**해 두었습니다. 아래 `RUN_PROFILE`만 바꾸면 됩니다.

`FINAL_SELECTION = 'AUTO_SSL'`은 **실제 SGL/NCL SSL을 사용한 모델 중 1위**로 최종 제출 CSV를 만듭니다. `AUTO_ALL`이면 SSL=0인 LightGCN 기준선도 포함합니다. `MAKE_SUBMISSION=True`는 Kaggle 서버에 자동 제출하는 기능이 아니라, **CSV 생성만 하는 기능**입니다.

학습 중단 이후 같은 Kaggle 실행 세션에서 재실행하면 구성·코드 해시가 일치하는 완료 실험은 건너뜁니다. 새 세션은 `/kaggle/working` 결과가 없어질 수 있으므로 `Save Version`으로 저장하세요.

In [ ]:
RUN_PROFILE = 'MAX'       # 'QUICK' (8개) | 'BALANCED' (20개 내외) | 'MAX' (전체)
MAKE_SUBMISSION = True    # True: 마지막에 원본 전체 train으로 재학습해 submission.csv 생성
FINAL_SELECTION = 'AUTO_SSL'  # 'AUTO_SSL' | 'AUTO_ALL' | 실험명 하나 (예: 'S11_dim128_ssl001')
SKIP_COMPLETED = True
BASELINE_REFERENCE = 0.03605490327712144  # 이전 LightGCN 검증 최고(참고선, 이번 후보 선정에 사용 안 함)

# 하이퍼파라미터 단위: ssl, proto는 곱해지는 loss weight; drop=엣지 제거 비율
# 같은 기본 설정에서 한 변수씩 바꾼 뒤 유망 조합을 별도 시험합니다.
def cfg(name, model, *, epochs=900, layers=3, dim=64, lr=.002,
        ssl=.005, proto=0., drop=.10, tau=.20, clusters=64,
        hyper=1, reg=.0001, ssl_nodes=192, samples=100000,
        warmup=100, cluster_every=50, alpha=1.):
    return dict(name=name, model=model, epochs=epochs, layers=layers, dim=dim,
                lr=lr, ssl_weight=ssl, proto_weight=proto, drop=drop, tau=tau,
                clusters=clusters, hyper_layers=hyper, reg=reg,
                ssl_nodes=ssl_nodes, samples_per_epoch=samples,
                warmup=warmup, cluster_every=cluster_every, alpha=alpha)

# SGL: 기존 ssl=0.02의 성능이 기준선보다 낮았으므로 작은 SSL과 강도 변경을 우선
ALL_EXPERIMENTS = [
    cfg('S00_baseline64_noSSL', 'sgl', epochs=900, ssl=0.),
    cfg('S01_ssl0005',         'sgl', epochs=900, ssl=.0005),
    cfg('S02_ssl001',          'sgl', epochs=900, ssl=.001),
    cfg('S03_ssl003',          'sgl', epochs=900, ssl=.003),
    cfg('S04_ssl005',          'sgl', epochs=900, ssl=.005),
    cfg('S05_ssl010',          'sgl', epochs=900, ssl=.01),
    cfg('S06_ssl020_repeat',   'sgl', epochs=700, ssl=.02),
    cfg('S07_drop005',         'sgl', epochs=900, ssl=.005, drop=.05),
    cfg('S08_drop020',         'sgl', epochs=900, ssl=.005, drop=.20),
    cfg('S09_drop030',         'sgl', epochs=900, ssl=.005, drop=.30),
    cfg('S10_tau010',          'sgl', epochs=900, ssl=.005, tau=.10),
    cfg('S11_tau040',          'sgl', epochs=900, ssl=.005, tau=.40),
    cfg('S12_dim128_noSSL',    'sgl', epochs=800, ssl=0., dim=128, lr=.003),
    cfg('S13_dim128_ssl0005',  'sgl', epochs=800, ssl=.0005, dim=128, lr=.003),
    cfg('S14_dim128_ssl001',   'sgl', epochs=800, ssl=.001, dim=128, lr=.003),
    cfg('S15_dim128_ssl003',   'sgl', epochs=800, ssl=.003, dim=128, lr=.003),
    cfg('S16_dim128_ssl005',   'sgl', epochs=800, ssl=.005, dim=128, lr=.003),
    cfg('S17_2layer_dim128',   'sgl', epochs=800, ssl=.001, dim=128, lr=.003, layers=2),
    cfg('S18_lr0015',          'sgl', epochs=900, ssl=.005, lr=.0015),
    cfg('S19_lr003',           'sgl', epochs=900, ssl=.005, lr=.003),
    cfg('S20_long1100',       'sgl', epochs=1100,ssl=.001, dim=128, lr=.003),
    cfg('S21_ssl_nodes384',    'sgl', epochs=900, ssl=.005, ssl_nodes=384),
    cfg('S22_samples200k',     'sgl', epochs=900, ssl=.005, samples=200000),
    cfg('S23_reg0005',        'sgl', epochs=900, ssl=.005, reg=.0005),

    # NCL: 700 Epoch에서 아직 상승 중. proto=.002가 악화되어 tiny weights 위주.
    cfg('N00_structure1000',    'ncl', epochs=1000,ssl=.005),
    cfg('N01_structure1400',    'ncl', epochs=1400,ssl=.005),
    cfg('N02_structure_ssl001', 'ncl', epochs=1100,ssl=.001),
    cfg('N03_structure_ssl002', 'ncl', epochs=1100,ssl=.002),
    cfg('N04_structure_ssl010', 'ncl', epochs=1100,ssl=.010),
    cfg('N05_dim128_lr003',     'ncl', epochs=1100,ssl=.005,dim=128,lr=.003),
    cfg('N06_layer2_dim64',    'ncl', epochs=1100,ssl=.005,layers=2),
    cfg('N07_layer2_dim128',   'ncl', epochs=1100,ssl=.005,layers=2,dim=128,lr=.003),
    cfg('N08_proto00005',      'ncl', epochs=1100,ssl=.005,proto=.00005),
    cfg('N09_proto0002',       'ncl', epochs=1100,ssl=.005,proto=.0002),
    cfg('N10_proto0005',       'ncl', epochs=1100,ssl=.005,proto=.0005),
    cfg('N11_proto_cluster128','ncl', epochs=1100,ssl=.005,proto=.0002,clusters=128),
    cfg('N12_proto_warmup300', 'ncl', epochs=1100,ssl=.005,proto=.0002,warmup=300),
    cfg('N13_hyper2',          'ncl', epochs=1100,ssl=.005,hyper=2),
    cfg('N14_tau010',          'ncl', epochs=1100,ssl=.005,tau=.10),
    cfg('N15_tau040',          'ncl', epochs=1100,ssl=.005,tau=.40),
    cfg('N16_dim128_ssl001',   'ncl', epochs=1100,ssl=.001,dim=128,lr=.003),
    cfg('N17_lr001',           'ncl', epochs=1100,ssl=.005,lr=.001),
]
PROFILES = {
    'QUICK': {
        'S00_baseline64_noSSL','S02_ssl001','S04_ssl005','S14_dim128_ssl001',
        'N00_structure1000','N03_structure_ssl002','N08_proto00005','N05_dim128_lr003',
    },
    'BALANCED': {
        'S00_baseline64_noSSL','S01_ssl0005','S02_ssl001','S03_ssl003','S04_ssl005',
        'S07_drop005','S08_drop020','S10_tau010','S11_tau040',
        'S12_dim128_noSSL','S13_dim128_ssl0005','S14_dim128_ssl001','S15_dim128_ssl003',
        'N00_structure1000','N01_structure1400','N02_structure_ssl001','N03_structure_ssl002',
        'N05_dim128_lr003','N06_layer2_dim64','N08_proto00005','N09_proto0002',
    },
    'MAX': {c['name'] for c in ALL_EXPERIMENTS}
}
assert len({c['name'] for c in ALL_EXPERIMENTS}) == len(ALL_EXPERIMENTS), '실험명 중복'
assert RUN_PROFILE in PROFILES
ACTIVE = [c for c in ALL_EXPERIMENTS if c['name'] in PROFILES[RUN_PROFILE]]
assert len(ACTIVE) == len(PROFILES[RUN_PROFILE])
assert all(c['model'] in ('sgl','ncl') for c in ACTIVE)
assert all(c['ssl_weight'] > 0 or c['proto_weight'] == 0 for c in ACTIVE)
print('PROFILE:',RUN_PROFILE,'| 실험 수:',len(ACTIVE),'| 총 Epoch:',sum(c['epochs'] for c in ACTIVE))
print('SUBMISSION:',MAKE_SUBMISSION,'| 선택:',FINAL_SELECTION)
for i,c in enumerate(ACTIVE,1):
    print(f'{i:02d}. {c["name"]:24s} {c["model"]} {c["epochs"]:4d}ep dim={c["dim"]} lr={c["lr"]} ssl={c["ssl_weight"]} proto={c["proto_weight"]}')

## 7. 실행·재시작·검증 함수

- 각 모델의 **완료 결과를 CSV/JSON/로그에 즉시 저장**합니다.
- 추천은 반드시 user별 **정확히 20개**. 이미 학습에 사용된 아이템 추천 금지.
- 점수는 **공통 teamkit.py**로 다시 계산해 모델 자체 점수와 일치하는지 검증합니다.
- 오류가 난 실험은 `failed_experiments.json`에 기록하고 나머지 실험을 계속합니다.

In [ ]:
from teamkit import read_interactions, read_truth, read_predictions, validate_predictions, recall_at_20
import pandas as pd

FIT_PATH = VALID/'val_train.csv'
USERS_PATH = VALID/'val_users.csv'
TRUTH_PATH = VALID/'val_truth.csv'
seen, popularity = read_interactions(FIT_PATH)
truth = read_truth(TRUTH_PATH)
catalog = set(popularity) | {i for items in truth.values() for i in items}
CODE_HASH = hashlib.sha256((WORK/'sgl_ncl_runner.py').read_bytes()).hexdigest()
GRAPH_HASH = hashlib.sha256((WORK/'graph_models.py').read_bytes()).hexdigest()
KIT_HASH = hashlib.sha256((WORK/'teamkit.py').read_bytes()).hexdigest()

assert len(truth) == manifest['validation_users'], '공통 검증 사용자 수 불일치'

def exp_paths(name):
    return {
      'pred':OUT/f'{name}_val.csv',
      'metrics':OUT/f'{name}.metrics.json',
      'config':OUT/f'{name}.config.json',
      'log':OUT/f'{name}.log',
      'checkpoint':OUT/f'{name}.pt'
    }

def stamp(c):
    return {'split_id':SPLIT_ID,'code_hash':CODE_HASH,'graph_hash':GRAPH_HASH,'kit_hash':KIT_HASH,'config':c}

def command(c, fit, users, out, metrics, truth_path=None, epochs=None):
    args=[sys.executable,str(WORK/'sgl_ncl_runner.py'),
          '--model',c['model'],'--fit',str(fit),'--users',str(users),
          '--out',str(out),'--metrics',str(metrics),
          '--device',DEVICE,'--seed',str(SEED),
          '--epochs',str(c['epochs'] if epochs is None else epochs),
          '--layers',str(c['layers']),'--dim',str(c['dim']),
          '--lr',str(c['lr']),'--reg',str(c['reg']),
          '--samples-per-epoch',str(c['samples_per_epoch']),
          '--ssl-nodes',str(c['ssl_nodes']),'--ssl-weight',str(c['ssl_weight']),
          '--drop',str(c['drop']),'--tau',str(c['tau']),
          '--proto-weight',str(c['proto_weight']),
          '--clusters',str(c['clusters']),'--hyper-layers',str(c['hyper_layers']),
          '--warmup',str(c['warmup']),'--cluster-every',str(c['cluster_every']),
          '--alpha',str(c['alpha']),
          '--eval-every','25','--patience','0','--eval-batch','256']
    if truth_path is not None:
        args.extend(['--truth',str(truth_path)])
    return args

def run_process(args, log, name):
    print(f'\n===== START {name} =====',flush=True)
    tail=[]
    log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('w',encoding='utf-8') as f:
        with subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as p:
            for line in p.stdout:
                f.write(line)
                if len(tail)>=35:tail.pop(0)
                tail.append(line.strip())
                try:
                    r=json.loads(line)
                    if 'epoch' in r and 'recall_at_20' in r:
                        print('Epoch',r['epoch'],'Recall@20',round(r['recall_at_20'],6),
                              'hits',r['hits'],'best',r.get('best'),flush=True)
                    elif 'final' in r:
                        print('FINAL',r['final'],flush=True)
                except (ValueError, TypeError):
                    if 'Traceback' in line or 'Error' in line:
                        print(line.strip(),flush=True)
            rc=p.wait()
    if rc:
        raise RuntimeError(f'{name} failed (exit={rc}):\n'+'\n'.join(tail))
    print(f'===== DONE {name} =====',flush=True)

def validate_exp(c):
    p=exp_paths(c['name'])
    if not all(p[k].is_file() for k in ('pred','metrics','config')):
        return None
    if json.loads(p['config'].read_text(encoding='utf-8')) != stamp(c):
        return None
    pred=read_predictions(p['pred'])
    validate_predictions(pred,truth,seen,catalog)
    score=recall_at_20(pred,truth)
    meta=json.loads(p['metrics'].read_text(encoding='utf-8'))
    if not meta.get('best_epoch') or abs(score['recall_at_20']-meta['recall_at_20'])>1e-12:
        raise AssertionError('점수 / best epoch 무결성 오류: '+c['name'])
    return {'experiment':c['name'],'model':c['model'],
            'ssl_on':(c['ssl_weight']>0 or c['proto_weight']>0),
            'best_epoch':int(meta['best_epoch']),
            **score,**{k:c[k] for k in ('epochs','layers','dim','lr','reg','ssl_weight',
             'proto_weight','drop','tau','clusters','hyper_layers','ssl_nodes','samples_per_epoch')},
            'split_id':SPLIT_ID}

def collect():
    good=[]
    for c in ACTIVE:
        try:
            r=validate_exp(c)
            if r is not None:good.append(r)
        except Exception as ex:
            print('INVALID',c['name'],str(ex)[:220])
    return sorted(good,key=lambda x:(-x['recall_at_20'],x['experiment']))

def save_summary():
    rows=collect()
    if rows:
        pd.DataFrame(rows).to_csv(OUT/'ablation_results.csv',index=False)
        (OUT/'ablation_results.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8')
    return rows

print('작동 준비 완료: SGL/NCL 소스 해시=',CODE_HASH[:12], 'fit=',FIT_PATH)

## 8. Ablation 전체 실행 (완료 즉시 저장)

`MAX`는 GPU 시간 제한에 따라 여러 시간 걸릴 수 있습니다. 오류 발생 시 나머지 실험은 계속하지만, **마지막에 실패 수를 반드시 확인**하세요. 중간에 세션이 중단되면 기존 Notebook 실행 환경에서 동일 코드 셀을 다시 실행하면 이미 완료된 실험은 건너뜁니다.

In [ ]:
failures={}
for idx,c in enumerate(ACTIVE,1):
    print(f'\n[{idx}/{len(ACTIVE)}] {c["name"]}',flush=True)
    p=exp_paths(c['name'])
    try:
        cached=validate_exp(c) if SKIP_COMPLETED else None
        if cached is not None:
            print('CACHE HIT: 검증된 결과 재사용',cached['recall_at_20'],flush=True)
            continue
        p['pred'].unlink(missing_ok=True)
        p['config'].unlink(missing_ok=True)
        run_process(command(c,FIT_PATH,USERS_PATH,p['pred'],p['metrics'],TRUTH_PATH),p['log'],c['name'])
        p['config'].write_text(json.dumps(stamp(c),ensure_ascii=False,indent=2),encoding='utf-8')
        score=validate_exp(c)
        if score is None:raise RuntimeError('결과 파일/점수 검증 실패')
        print('SCORE',round(score['recall_at_20'],8),'hits',score['hits'],'best epoch',score['best_epoch'],flush=True)
    except Exception as ex:
        failures[c['name']]=str(ex)
        print('EXPERIMENT FAILED:',c['name'],str(ex)[-650:],flush=True)
    rows=save_summary()
    if rows:
        print(f'현재 완료 {len(rows)}/{len(ACTIVE)} | 최고: {rows[0]["experiment"]} Recall@20={rows[0]["recall_at_20"]:.6f}',flush=True)
(OUT/'failed_experiments.json').write_text(json.dumps(failures,ensure_ascii=False,indent=2),encoding='utf-8')
print('\nABLATION FINISHED', '성공:',len(collect()), '/',len(ACTIVE),'실패:',len(failures))

## 9. 전체 Ablation 순위표 및 SGL·NCL별 최고 성능

In [ ]:
from IPython.display import display
rows=save_summary()
if not rows:
    raise RuntimeError('완료된 실험이 없습니다. 8번 실행 로그와 failed_experiments.json을 확인하세요.')
comparison=pd.DataFrame(rows)
display(comparison[['experiment','model','ssl_on','recall_at_20','hits','best_epoch',
                   'layers','dim','lr','ssl_weight','proto_weight','drop','tau']])
print('이전 LightGCN 최고(참고):',BASELINE_REFERENCE)
print('이번 최고:',rows[0]['experiment'],rows[0]['recall_at_20'])
for name in ('sgl','ncl'):
    subset=[r for r in rows if r['model']==name and r['ssl_on']]
    if subset:
        print(name.upper(),'SSL 사용 최고:',subset[0]['experiment'],subset[0]['recall_at_20'])
print('결과 CSV:',OUT/'ablation_results.csv')
print('실패 기록:',OUT/'failed_experiments.json')

## 10. 이번 Ablation에서 제출할 모델 하나 선택 → **전체 train으로 재학습**

- `FINAL_SELECTION='AUTO_SSL'`: 실제 SSL을 사용한 SGL·NCL 중 Validation 1위 (기본값)
- `FINAL_SELECTION='AUTO_ALL'`: SSL을 끈 LightGCN 기준선을 포함한 전체 1위
- `FINAL_SELECTION='S14_dim128_ssl001'`: 특정 실험명을 수동 지정

**선택된 best_epoch로 전체 train.csv를 새 모델로 처음부터 학습합니다.** validation의 `val_train.csv`만 사용하는 것이 아닙니다. `test.csv`의 모든 사용자에게 정확히 Top-20을 추천하고 팀 공통 제출 형식 검사를 통과해야 CSV를 저장합니다. Kaggle 대회로 자동 제출하는 것은 아닙니다.

노트북 실행 결과에서 `SUBMISSION READY`가 표시되기 전에는 CSV를 제출하지 마세요.

In [ ]:
if not MAKE_SUBMISSION:
    print('MAKE_SUBMISSION=False: 검증 실험만 완료. CSV를 만들려면 6번 셀을 True로 바꾸고 이 셀을 다시 실행하세요.')
else:
    rows=save_summary()
    if not rows:
        raise RuntimeError('Validation 결과 없음')
    if FINAL_SELECTION == 'AUTO_SSL':
        candidates=[r for r in rows if r['ssl_on']]
        if not candidates:raise RuntimeError('SSL을 켠 SGL/NCL 실험 결과가 없습니다.')
        selected=candidates[0]
    elif FINAL_SELECTION == 'AUTO_ALL':
        selected=rows[0]
    else:
        candidates=[r for r in rows if r['experiment']==FINAL_SELECTION]
        if len(candidates)!=1:
            raise ValueError('선택한 실험명을 결과표에서 찾지 못했습니다: '+FINAL_SELECTION)
        selected=candidates[0]
    c=next(x for x in ACTIVE if x['name']==selected['experiment'])
    selected_epoch=int(selected['best_epoch'])
    assert selected_epoch>0
    final_csv=WORK/'submission.csv'
    final_metrics=OUT/'final_submission.metrics.json'
    print('FINAL SELECTED:',c['name'], '| SSL:',selected['ssl_on'],
          '| Validation Recall@20:',selected['recall_at_20'],
          '| FULL TRAIN epochs:',selected_epoch,'| fit=',TRAIN,flush=True)
    final_csv.unlink(missing_ok=True)
    (OUT/'selected_final_config.json').unlink(missing_ok=True)
    run_process(command(c,TRAIN,TEST,final_csv,final_metrics,truth_path=None,epochs=selected_epoch),
                OUT/'final_submission.log','FINAL '+c['name'])
    subprocess.run([sys.executable,str(WORK/'teamkit.py'),'check',
                    '--pred',str(final_csv),'--train',str(TRAIN),'--test',str(TEST)],check=True)
    selected_record={
        'experiment':c['name'],'model':c['model'],'config':c,
        'best_epoch_from_validation':selected_epoch,
        'validation_recall_at_20':selected['recall_at_20'],
        'split_id':SPLIT_ID,'full_train':str(TRAIN),'test_users':str(TEST),
        'submission':str(final_csv),'submission_filename':'submission.csv',
        'selection_rule':FINAL_SELECTION
    }
    (OUT/'selected_final_config.json').write_text(json.dumps(selected_record,ensure_ascii=False,indent=2),encoding='utf-8')
    from IPython.display import display
    df=pd.read_csv(final_csv)
    assert list(df.columns)==['user_id','item_id']
    assert len(df)==pd.read_csv(TEST).shape[0]
    assert df['item_id'].map(lambda x:len(str(x).split())==20).all()
    display(df.head(3))
    print('SUBMISSION READY:',final_csv)

## 11. 다운로드할 ZIP 만들기

결과 ZIP은 **대회 원본 데이터나 Validation 정답을 포함하지 않고**, 실험 점수/로그/설정과 최종 제출용 CSV만 포함합니다. Kaggle UI에서 `FileLink`를 눌렀을 때 404가 나면 `/kaggle/working/sgl_ncl_extended_submission.zip`을 Notebook Output의 저장된 버전에서 다운로드하세요. **Kaggle에는 ZIP이 아니라 압축 해제한 `submission.csv`를 제출**해야 합니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
bundle=WORK/'sgl_ncl_extended_submission.zip'
with ZipFile(bundle,'w',ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        # 40개의 Top-20 CSV와 체크포인트는 ZIP 용량을 크게 하므로 제외 (원본 파일은 working에 남아 있음)
        if p.is_file() and p.suffix in ('.json','.log') or (p.is_file() and p.name=='ablation_results.csv'):
            z.write(p,arcname='results/'+p.name)
    final=WORK/'submission.csv'
    if final.is_file() and (OUT/'selected_final_config.json').is_file():
        z.write(final,arcname='submission.csv')
print('결과 ZIP:',bundle, '| bytes:',bundle.stat().st_size)
print('선택 모델:',(OUT/'selected_final_config.json').read_text()[:600] if (OUT/'selected_final_config.json').is_file() else '아직 최종 제출 없음')